# ComfyUI on Colab — Personalised Launcher\n\nTwo-cell launcher. Run Cell 1, pick pipelines in the widget, click **Save selection**, then run Cell 2.

In [ ]:
# ====================================================================
# CELL 1 — SETUP + PIPELINE SELECTOR
# Run this cell, wait for the widget, pick pipelines + LoRAs,
# then click "Save selection" before running Cell 2.
# ====================================================================

import os, subprocess, sys, getpass

# --- Drive (soft-fail; used only for a ~1KB prefs.json) -------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PREFS_PATH = "/content/drive/MyDrive/.comfyui-colab/prefs.json"
except Exception:
    PREFS_PATH = "/content/.comfyui-colab-prefs.json"

# --- GPU detect + conditional torch reinstall ------------------------
def _cc():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            text=True).strip().splitlines()[0]
        name, cc = [s.strip() for s in out.split(",", 1)]
        maj, mn = cc.split(".")
        return name, int(maj) * 10 + int(mn)
    except Exception:
        return "CPU", 0

GPU_NAME, GPU_CC = _cc()
print(f"GPU: {GPU_NAME} (compute_cap={GPU_CC})")

import torch
arches = torch.cuda.get_arch_list() if torch.cuda.is_available() else []
max_arch = max((int(a.split("_")[-1]) for a in arches if a.startswith("sm_")), default=0)
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} max_arch=sm_{max_arch}")
if GPU_CC >= 120 and max_arch < 120:
    print("Reinstalling torch for cu128…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu128"])
    print("Reinstalled. RESTART runtime, then re-run this cell.")
    raise SystemExit(0)

# --- Secrets ---------------------------------------------------------
def _load(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    try:
        v = getpass.getpass(f"{name} (hidden, Enter to skip): ")
        return v.strip() or None
    except Exception:
        return None

for k in ("HF_TOKEN", "CIVITAI_TOKEN"):
    v = _load(k)
    if v:
        os.environ[k] = v
        print(f"{k}: loaded")
    else:
        print(f"{k}: skipped")

# --- Deps ------------------------------------------------------------
subprocess.check_call(["apt-get", "install", "-y", "-qq", "aria2"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "ipywidgets", "huggingface_hub", "huggingface_hub[cli]"])

# --- Clone repo ------------------------------------------------------
REPO_URL = os.environ.get("COMFYUI_COLAB_REPO_URL",
                          "https://github.com/SamuelD27/ComfyCustom.git")
REPO_BRANCH = os.environ.get("COMFYUI_COLAB_BRANCH", "Collab")
REPO_DIR = "/content/ComfyUI"
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo ready at {REPO_DIR} (branch: {REPO_BRANCH})")

# --- Selector UI -----------------------------------------------------
import ipywidgets as W
from IPython.display import display
from colab.launcher_helpers import (
    load_registry, list_pipelines, pipeline_loras, load_prefs, save_prefs,
)

REGISTRY_PATH = f"{REPO_DIR}/scripts/model_registry.json"
registry = load_registry(REGISTRY_PATH)
pipelines = list_pipelines(registry)
prefs = load_prefs(PREFS_PATH)
sel_prev = set(prefs.get("pipelines", []))
lora_prev = {k: set(v) for k, v in prefs.get("loras", {}).items()}

STATE = {"pipelines": set(), "loras": {}, "prefs_path": PREFS_PATH,
         "registry": registry, "repo_dir": REPO_DIR, "ready": False}

pipeline_boxes = []
for p in pipelines:
    cb = W.Checkbox(
        value=(p["key"] in sel_prev),
        description=f"{p['display_name']}  ({p['total_size_gb']} GB, "
                    f"{p['model_count']} models, {p['lora_count']} LoRAs)",
        indent=False, layout=W.Layout(width="100%"))
    pipeline_boxes.append((p["key"], cb))

lora_pane = W.VBox([], layout=W.Layout(width="100%"))

def _update_lora(*_):
    STATE["pipelines"] = {k for k, cb in pipeline_boxes if cb.value}
    kids, STATE["loras"] = [], {}
    for key in sorted(STATE["pipelines"]):
        loras = pipeline_loras(registry, key)
        if not loras:
            continue
        opts = [(f"{l['display_name']} ({l.get('size_gb', 0):.2f} GB)", l["filename"])
                for l in loras]
        pre = [fn for _d, fn in opts if fn in lora_prev.get(key, set())]
        sm = W.SelectMultiple(options=opts, value=tuple(pre), description=key,
                              rows=min(6, len(opts)),
                              layout=W.Layout(width="100%"))
        STATE["loras"][key] = set(pre)
        def _h(change, _k=key): STATE["loras"][_k] = set(change["new"])
        sm.observe(_h, names="value")
        kids.append(sm)
    lora_pane.children = kids

for _, cb in pipeline_boxes:
    cb.observe(_update_lora, names="value")

save_btn = W.Button(description="Save selection",
                    button_style="success", icon="check")
status = W.HTML("<i>Pick pipelines + LoRAs, then click <b>Save selection</b>. "
                "Then run Cell 2.</i>")

def _save(_b):
    if not STATE["pipelines"]:
        status.value = "<span style='color:#a00'>Select at least one pipeline.</span>"
        return
    save_prefs(PREFS_PATH, {
        "pipelines": sorted(STATE["pipelines"]),
        "loras": {k: sorted(v) for k, v in STATE["loras"].items()},
    })
    STATE["ready"] = True
    n_loras = sum(len(v) for v in STATE["loras"].values())
    status.value = (f"<b>Saved.</b> {len(STATE['pipelines'])} pipelines, "
                    f"{n_loras} LoRAs. Now run Cell 2.")

save_btn.on_click(_save)
_update_lora()
display(W.VBox([
    W.HTML("<h3>Pipelines</h3>"),
    W.VBox([cb for _, cb in pipeline_boxes]),
    W.HTML("<h3>LoRAs (per selected pipeline)</h3>"),
    lora_pane,
    save_btn, status,
]))


In [ ]:
# ====================================================================
# CELL 2 — DOWNLOAD WEIGHTS + LAUNCH COMFYUI + PUBLIC URL
# Run this AFTER clicking "Save selection" in Cell 1.
# ====================================================================

import os, subprocess, sys, socket, threading, time, urllib.request
from pathlib import Path
from colab.launcher_helpers import (
    pipeline_models, pipeline_loras,
    build_hf_download_cmd, build_aria2c_cmd, write_auth_header,
    extract_trycloudflare_url,
)

if not STATE.get("ready"):
    raise SystemExit("Click 'Save selection' in Cell 1 first.")

REPO_DIR = STATE["repo_dir"]
registry = STATE["registry"]
MODELS_ROOT = Path(REPO_DIR) / "models"

# --- Install ComfyUI + custom-node requirements ----------------------
print("Installing ComfyUI requirements…")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                       f"{REPO_DIR}/requirements.txt"])
for cn in Path(f"{REPO_DIR}/custom_nodes").iterdir():
    req = cn / "requirements.txt"
    if req.exists():
        print(f"  custom_nodes/{cn.name}/requirements.txt")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "-r", str(req)])

# --- Download weights ------------------------------------------------
auth_hdr = None
if os.environ.get("CIVITAI_TOKEN"):
    auth_hdr = write_auth_header(os.environ["CIVITAI_TOKEN"])

for pkey in sorted(STATE["pipelines"]):
    print(f"\n=== {pkey} ===")
    for m in pipeline_models(registry, pkey):
        dest_dir = MODELS_ROOT / m["subdir"]
        dest_dir.mkdir(parents=True, exist_ok=True)
        target = dest_dir / m["filename"]
        if target.exists():
            print(f"  skip (exists): {m['filename']}")
            continue
        print(f"  HF: {m['filename']} ({m.get('size_gb', 0):.1f} GB)")
        cmd = build_hf_download_cmd(m, dest_dir, os.environ.get("HF_TOKEN"))
        subprocess.check_call(cmd)
    for l in pipeline_loras(registry, pkey):
        if l["filename"] not in STATE["loras"].get(pkey, set()):
            continue
        dest_dir = MODELS_ROOT / "loras"
        dest_dir.mkdir(parents=True, exist_ok=True)
        target = dest_dir / l["filename"]
        if target.exists():
            print(f"  skip LoRA (exists): {l['filename']}")
            continue
        print(f"  LoRA: {l['filename']} ({l.get('size_gb', 0):.2f} GB)")
        cmd = build_aria2c_cmd(l, dest_dir, auth_hdr)
        subprocess.check_call(cmd)

# --- Launch ComfyUI --------------------------------------------------
LOG = "/content/comfyui.log"
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd=REPO_DIR,
    stdout=open(LOG, "a"),
    stderr=subprocess.STDOUT,
)
print(f"\nComfyUI started (pid={comfy_proc.pid}). Logs: {LOG}")

# --- Cloudflared tunnel ---------------------------------------------
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Downloading cloudflared…")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared")
    os.chmod("/usr/local/bin/cloudflared", 0o755)

PORT = 8188
PUBLIC_URL = {"url": None}

def _tunnel():
    for _ in range(300):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", PORT)) == 0:
                break
        time.sleep(1)
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for line in proc.stdout:
        if PUBLIC_URL["url"] is None:
            url = extract_trycloudflare_url(line)
            if url:
                PUBLIC_URL["url"] = url
                print(f"\n*** PUBLIC URL: {url} ***\n")

threading.Thread(target=_tunnel, daemon=True).start()

# Wait up to 5 min for the URL
print("Waiting for public URL…")
for _ in range(300):
    if PUBLIC_URL["url"]:
        break
    time.sleep(1)
print(f"Final URL: {PUBLIC_URL['url']}")
